# Day 24 — Feature Engineering & sklearn Pipelines

**Dataset:** `telecom_churn.csv`


**Main goal:** Build leakage-proof preprocessing using `Pipeline` and `ColumnTransformer`.


In [2]:
# SETUP

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import (
    OneHotEncoder, OrdinalEncoder,
    StandardScaler, MinMaxScaler, RobustScaler,
    PolynomialFeatures
)
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.base import BaseEstimator, TransformerMixin

# Load data
df = pd.read_csv("telecom_churn.csv")

print("Shape:", df.shape)
print("\nData types:")
print(df.dtypes)

# Target and ID
X = df.drop(columns=["churn", "customer_id"]).copy()
y = df["churn"].copy()

# Fixed split requested by the task sheet
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

# Columns from the task sheet
categorical_cols = ["contract_type", "payment_method", "region"]
date_col = "signup_date"

# Detect numeric columns automatically
numeric_cols = X_train.select_dtypes(include=np.number).columns.tolist()

print("\nNumeric columns:", numeric_cols)
print("Categorical columns:", categorical_cols)
print("Date column:", date_col)
print("\nMissing values:")
print(df.isna().sum())


Shape: (2000, 10)

Data types:
customer_id            str
tenure_months        int64
age                  int64
monthly_charges    float64
total_charges      float64
contract_type          str
payment_method         str
region                 str
signup_date            str
churn                int64
dtype: object

Numeric columns: ['tenure_months', 'age', 'monthly_charges', 'total_charges']
Categorical columns: ['contract_type', 'payment_method', 'region']
Date column: signup_date

Missing values:
customer_id         0
tenure_months       0
age                 0
monthly_charges     0
total_charges      40
contract_type       0
payment_method      0
region              0
signup_date         0
churn               0
dtype: int64



- `customer_id` is dropped because it is an identifier.
- `churn` is the binary target.
- Numeric, categorical, and date columns are handled separately.
- `SimpleImputer` is used inside preprocessing pipelines so missing-value statistics are learned from training data only.
- `random_state=42` makes the comparison reproducible.


In [3]:
# TASK 01 — One-hot, ordinal, and target encoding

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# -----------------------------
# 1. One-hot encoding
# -----------------------------
onehot_pipe = Pipeline([
    ("prep", ColumnTransformer([
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), numeric_cols),

        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore"))
        ]), categorical_cols)
    ])),
    ("model", LogisticRegression(max_iter=2000))
])

# -----------------------------
# 2. Ordinal encoding
# -----------------------------
ordinal_pipe = Pipeline([
    ("prep", ColumnTransformer([
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), numeric_cols),

        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OrdinalEncoder(
                handle_unknown="use_encoded_value",
                unknown_value=-1
            ))
        ]), categorical_cols)
    ])),
    ("model", LogisticRegression(max_iter=2000))
])

# -----------------------------
# 3. Target encoding
# -----------------------------
def target_encode_train_test(X_tr, y_tr, X_va):
    X_tr = X_tr.copy()
    X_va = X_va.copy()

    # Fill missing categories using training-fold mode
    for col in categorical_cols:
        mode = X_tr[col].mode()
        fill_value = mode.iloc[0] if len(mode) else "MISSING"

        X_tr[col] = X_tr[col].fillna(fill_value)
        X_va[col] = X_va[col].fillna(fill_value)

    # Global target mean from TRAINING fold only
    global_mean = y_tr.mean()

    for col in categorical_cols:
        mapping = (
            pd.DataFrame({
                "category": X_tr[col],
                "target": y_tr.values
            })
            .groupby("category")["target"]
            .mean()
        )

        # Learn mappings only from training fold
        X_tr[col] = X_tr[col].map(mapping)
        X_va[col] = X_va[col].map(mapping)

        # Unseen validation categories use training global mean
        X_tr[col] = X_tr[col].fillna(global_mean)
        X_va[col] = X_va[col].fillna(global_mean)

    # Numeric missing values: training-fold medians only
    for col in numeric_cols:
        median_value = X_tr[col].median()
        X_tr[col] = X_tr[col].fillna(median_value)
        X_va[col] = X_va[col].fillna(median_value)

    return X_tr, X_va


target_scores = []

for train_idx, valid_idx in cv.split(X_train, y_train):

    X_tr = X_train.iloc[train_idx].copy()
    X_va = X_train.iloc[valid_idx].copy()
    y_tr = y_train.iloc[train_idx]
    y_va = y_train.iloc[valid_idx]

    X_tr_te, X_va_te = target_encode_train_test(
        X_tr, y_tr, X_va
    )

    scaler = StandardScaler()

    X_tr_num = scaler.fit_transform(X_tr_te[numeric_cols])
    X_va_num = scaler.transform(X_va_te[numeric_cols])

    X_tr_cat = X_tr_te[categorical_cols].values
    X_va_cat = X_va_te[categorical_cols].values

    Xtr_final = np.hstack([X_tr_num, X_tr_cat])
    Xva_final = np.hstack([X_va_num, X_va_cat])

    # Confirm no missing values reach LogisticRegression
    assert not np.isnan(Xtr_final).any()
    assert not np.isnan(Xva_final).any()

    model = LogisticRegression(max_iter=2000)
    model.fit(Xtr_final, y_tr)

    target_scores.append(model.score(Xva_final, y_va))


onehot_score = cross_val_score(
    onehot_pipe, X_train, y_train,
    cv=cv, scoring="accuracy"
).mean()

ordinal_score = cross_val_score(
    ordinal_pipe, X_train, y_train,
    cv=cv, scoring="accuracy"
).mean()

target_score = np.mean(target_scores)

encoding_results = pd.DataFrame({
    "Encoding": ["One-hot", "Ordinal", "Target (mean)"],
    "Mean CV Accuracy": [
        onehot_score,
        ordinal_score,
        target_score
    ]
}).sort_values("Mean CV Accuracy", ascending=False)

print("Encoding comparison:")
print(encoding_results.to_string(index=False))


Encoding comparison:
     Encoding  Mean CV Accuracy
Target (mean)          0.773333
      Ordinal          0.771333
      One-hot          0.760667


## Task 01 — Encoding comparison

**One-hot:** Usually best for nominal categories with low/moderate cardinality because it does not create a false order.

**Ordinal:** Appropriate when categories have a real order such as low < medium < high. For nominal categories it can create an artificial numeric relationship.

**Target encoding:** Useful for high-cardinality categories because each category becomes one numeric target-related value. It must be fitted using training data only.

### Look it up — answers

**1. Why is target encoding leaking if the category mean is computed on the full dataset?**  
Because validation/test target values are included in the category mean. The feature therefore contains information about the answer being evaluated.

**2. When do people use one-hot vs ordinal vs target encoding?**  
One-hot is common for low/moderate-cardinality nominal data; ordinal is for genuinely ordered categories; target encoding can be useful for high-cardinality categories.

**3. What happens if a category appears only once in training?**  
Its target mean is exactly that one observation's target, making the estimate unstable and prone to overfitting. Smoothing or a global-mean fallback can help.

**Why can the CV scores differ?**  
The encodings give the same categories different numerical representations, which changes what LogisticRegression can learn.

**Submission evidence:** The code prints the three actual CV scores.


In [4]:
# TASK 02 — Standard, Min-Max, and Robust scaling

X_num = X_train[numeric_cols].copy()

print("Numeric summary:")
print(X_num.describe().T)

print("\nPotential outliers using the IQR rule:")

for col in numeric_cols:
    q1 = X_num[col].quantile(0.25)
    q3 = X_num[col].quantile(0.75)
    iqr = q3 - q1

    outliers = (
        (X_num[col] < q1 - 1.5 * iqr) |
        (X_num[col] > q3 + 1.5 * iqr)
    ).sum()

    print(f"{col}: {outliers}")

if "monthly_charges" in X_num.columns:
    print("\nmonthly_charges quantiles:")
    print(X_num["monthly_charges"].quantile([0, .25, .5, .75, 1]))

if "total_charges" in X_num.columns:
    print("\ntotal_charges quantiles:")
    print(X_num["total_charges"].quantile([0, .25, .5, .75, 1]))


scalers = {
    "StandardScaler": StandardScaler(),
    "MinMaxScaler": MinMaxScaler(),
    "RobustScaler": RobustScaler()
}

scaler_scores = {}

for name, scaler in scalers.items():

    pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", scaler),
        ("model", LogisticRegression(max_iter=2000))
    ])

    scores = cross_val_score(
        pipe,
        X_num,
        y_train,
        cv=cv,
        scoring="accuracy"
    )

    scaler_scores[name] = scores.mean()


scaler_results = pd.DataFrame({
    "Scaler": scaler_scores.keys(),
    "Mean CV Accuracy": scaler_scores.values()
}).sort_values("Mean CV Accuracy", ascending=False)

print("\nScaler comparison:")
print(scaler_results.to_string(index=False))


Numeric summary:
                  count         mean          std    min      25%      50%  \
tenure_months    1500.0    21.760000    20.078957   1.00    6.000   15.000   
age              1500.0    41.874667    13.048645  18.00   33.000   42.000   
monthly_charges  1500.0    64.147293    24.776000  18.00   47.100   63.245   
total_charges    1467.0  1407.433327  1502.655464  17.73  307.105  883.280   

                      75%      max  
tenure_months      31.000    72.00  
age                51.000    85.00  
monthly_charges    80.935   150.00  
total_charges    1944.175  8337.18  

Potential outliers using the IQR rule:
tenure_months: 79
age: 4
monthly_charges: 10
total_charges: 89

monthly_charges quantiles:
0.00     18.000
0.25     47.100
0.50     63.245
0.75     80.935
1.00    150.000
Name: monthly_charges, dtype: float64

total_charges quantiles:
0.00      17.730
0.25     307.105
0.50     883.280
0.75    1944.175
1.00    8337.180
Name: total_charges, dtype: float64

Scaler com

## Task 02 — Scaling comparison

**StandardScaler:** Centers values around 0 and scales using standard deviation. It is sensitive to extreme values.

**MinMaxScaler:** Maps values to a fixed range, usually 0–1. Extreme values can compress normal observations into a small range.

**RobustScaler:** Uses the median and IQR, so it is much less affected by outliers.

### Look it up — answers

**1. What does RobustScaler do differently from StandardScaler?**  
RobustScaler uses the median and interquartile range instead of the mean and standard deviation.

**2. When can MinMaxScaler be harmful?**  
When strong outliers exist, the extreme minimum/maximum can dominate the scaling and compress most observations.

**3. Why does scaling matter more for Ridge/LogisticRegression than trees?**  
Linear models use feature magnitudes during optimization and regularization. Trees split using thresholds and are largely unaffected by rescaling.

**When does RobustScaler beat the others?**  
When the numeric data contains substantial outliers that should not dominate the scaling.

**Submission evidence:** Outlier information and actual CV scores are printed by the code.


In [5]:
# TASK 03 — Polynomial and interaction features

preferred_poly = [
    c for c in ["monthly_charges", "total_charges", "tenure"]
    if c in numeric_cols
]

poly_cols = (
    preferred_poly
    if len(preferred_poly) >= 2
    else numeric_cols[:min(3, len(numeric_cols))]
)

X_base = X_train[poly_cols].copy()

base_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
    ("model", LogisticRegression(max_iter=3000))
])

poly_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("poly", PolynomialFeatures(degree=2, include_bias=False)),
    ("scale", StandardScaler()),
    ("model", LogisticRegression(max_iter=3000))
])

interaction_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("poly", PolynomialFeatures(
        degree=2,
        interaction_only=True,
        include_bias=False
    )),
    ("scale", StandardScaler()),
    ("model", LogisticRegression(max_iter=3000))
])

base_cv = cross_val_score(
    base_model, X_base, y_train,
    cv=cv, scoring="accuracy"
).mean()

poly_cv = cross_val_score(
    poly_model, X_base, y_train,
    cv=cv, scoring="accuracy"
).mean()

interaction_cv = cross_val_score(
    interaction_model, X_base, y_train,
    cv=cv, scoring="accuracy"
).mean()

comparison = pd.DataFrame({
    "Feature set": [
        "Original",
        "Polynomial degree 2",
        "Interaction only"
    ],
    "Mean CV Accuracy": [
        base_cv,
        poly_cv,
        interaction_cv
    ]
})

comparison["Change vs Original"] = (
    comparison["Mean CV Accuracy"] - base_cv
)

print("Polynomial/interaction comparison:")
print(comparison.to_string(index=False))

# Strongest generated terms
imputed_base = SimpleImputer(strategy="median").fit_transform(X_base)

poly = PolynomialFeatures(
    degree=2,
    include_bias=False
)

X_poly_arr = poly.fit_transform(imputed_base)
poly_names = poly.get_feature_names_out(poly_cols)

poly_df = pd.DataFrame(
    X_poly_arr,
    columns=poly_names,
    index=X_base.index
)

corrs = (
    poly_df.join(y_train.rename("target"))
    .corr(numeric_only=True)["target"]
    .drop("target")
    .abs()
    .sort_values(ascending=False)
)

print("\nStrongest generated terms by absolute target correlation:")
print(corrs.head(5))

# Ridge check
ridge_poly = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("poly", PolynomialFeatures(degree=2, include_bias=False)),
    ("scale", StandardScaler()),
    ("model", Ridge(alpha=1.0))
])

ridge_cv = cross_val_score(
    ridge_poly,
    X_base,
    y_train,
    cv=cv,
    scoring="r2"
).mean()

print("\nRidge polynomial CV R2:", ridge_cv)


Polynomial/interaction comparison:
        Feature set  Mean CV Accuracy  Change vs Original
           Original             0.764               0.000
Polynomial degree 2             0.770               0.006
   Interaction only             0.764               0.000

Strongest generated terms by absolute target correlation:
monthly_charges                  0.164116
monthly_charges^2                0.148941
total_charges                    0.059994
total_charges^2                  0.059372
monthly_charges total_charges    0.014574
Name: target, dtype: float64

Ridge polynomial CV R2: 0.04193056743518908


## Task 03 — Polynomial and interaction features

The comparison table gives evidence instead of relying on visual intuition.

**Verdict:**  
- If polynomial/interaction CV accuracy is higher than the original score, the added features helped.
- If it is lower, the extra features did not improve generalization.
- The strongest generated terms are printed as candidates; correlation alone does not prove causation.

### Look it up — answers

**1. What does `interaction_only=True` do?**  
It creates interaction products such as `x1*x2` but excludes powers such as `x1²`.

**2. Why can polynomial features make LinearRegression unstable, and how does Ridge help?**  
Polynomial features can create highly correlated predictors. This multicollinearity can make coefficients unstable. Ridge adds an L2 penalty that stabilizes coefficients and reduces overfitting.

**3. How do you choose polynomial degree without silently overfitting?**  
Treat degree as a hyperparameter and select it using cross-validation inside the training process. Do not select it based on the test set.

**Submission evidence:** Original vs polynomial vs interaction CV scores and Ridge results are printed.


In [ ]:
# TASK 04 — Turn signup_date into usable features

df_date = df.copy()

df_date["signup_date"] = pd.to_datetime(
    df_date["signup_date"],
    errors="coerce"
)

# Required date components
df_date["signup_year"] = df_date["signup_date"].dt.year
df_date["signup_month"] = df_date["signup_date"].dt.month
df_date["signup_dayofweek"] = df_date["signup_date"].dt.dayofweek
df_date["signup_quarter"] = df_date["signup_date"].dt.quarter

# Training-only reference date
train_reference_date = df_date.loc[
    X_train.index,
    "signup_date"
].max()

df_date["days_since"] = (
    train_reference_date - df_date["signup_date"]
).dt.days

date_features = [
    "signup_year",
    "signup_month",
    "signup_dayofweek",
    "signup_quarter",
    "days_since"
]

print("Date feature sample:")
print(df_date[date_features].head())

# Training data
X_date_train = df_date.loc[
    X_train.index,
    numeric_cols + date_features
].copy()

base_date_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
    ("model", LogisticRegression(max_iter=3000))
])

date_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
    ("model", LogisticRegression(max_iter=3000))
])

base_date_cv = cross_val_score(
    base_date_model,
    X_date_train[numeric_cols],
    y_train,
    cv=cv,
    scoring="accuracy"
).mean()

date_cv = cross_val_score(
    date_model,
    X_date_train[numeric_cols + date_features],
    y_train,
    cv=cv,
    scoring="accuracy"
).mean()

print("\nBefore date features:", base_date_cv)
print("After date features :", date_cv).
print("Change              :", date_cv - base_date_cv)


Date feature sample:
   signup_year  signup_month  signup_dayofweek  signup_quarter  days_since
0         2022             8                 2               3         515
1         2022             8                 0               3         510
2         2023             3                 4               1         289
3         2021            10                 1               4         810
4         2023             6                 4               2         205

Before date features: 0.7673333333333333
After date features : 0.7666666666666667
Change              : -0.0006666666666665932


## Task 04 — Date feature engineering

The raw date is converted into:
- year
- month
- day of week
- quarter
- days since a training-derived reference date

### Look it up — answers

**1. Why should you almost never feed a raw date string to a model?**  
A raw date string does not directly expose useful time structure such as month, weekday, elapsed time, or season.

**2. Which derived date features are likely to matter for churn?**  
Elapsed time/tenure-related features, signup period, month/season, and potentially weekday can capture differences in customer behavior over time.

**3. Why must the `days_since` reference date come from training data only?**  
Using a reference calculated from the full dataset can use information from the held-out/test period. A training-only reference keeps feature creation leakage-safe.

**Submission evidence:** The code prints the engineered date features and before/after CV scores.


In [7]:
# TASK 05 — Spot the leakage and fix it

# ---------------------------------------------------------
# DELIBERATELY LEAKY VERSION
# ---------------------------------------------------------
X_num_all = X[numeric_cols].copy()

leaky_scaler = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# This is deliberately wrong:
# preprocessing is fitted on ALL rows before the split.
X_all_scaled = leaky_scaler.fit_transform(X_num_all)

X_train_leaky, X_test_leaky, y_train_leaky, y_test_leaky = train_test_split(
    X_all_scaled,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

leaky_model = LogisticRegression(max_iter=2000)

# The scaling has already seen the future held-out rows.
leaky_scores = cross_val_score(
    leaky_model,
    X_train_leaky,
    y_train_leaky,
    cv=cv,
    scoring="accuracy"
)

# ---------------------------------------------------------
# FIXED VERSION
# ---------------------------------------------------------
fixed_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=2000))
])

fixed_scores = cross_val_score(
    fixed_pipe,
    X_train[numeric_cols],
    y_train,
    cv=cv,
    scoring="accuracy"
)

results = pd.DataFrame({
    "Version": [
        "Leaky preprocessing",
        "Fixed Pipeline preprocessing"
    ],
    "Mean CV Accuracy": [
        leaky_scores.mean(),
        fixed_scores.mean()
    ],
    "Std CV Accuracy": [
        leaky_scores.std(),
        fixed_scores.std()
    ]
})

print(results.to_string(index=False))

print(
    "\nLeaky - fixed score difference:",
    leaky_scores.mean() - fixed_scores.mean()
)


                     Version  Mean CV Accuracy  Std CV Accuracy
         Leaky preprocessing          0.767333         0.003887
Fixed Pipeline preprocessing          0.767333         0.003887

Leaky - fixed score difference: 0.0


## Task 05 — Leakage

**Exact leak:** The scaler/imputer is fitted before the train/test split. Therefore its learned statistics can include information from rows that later become the held-out test set.

For scaling, the learned statistics are the **mean and standard deviation**. For imputation, the learned statistic can be the **median**.

### Look it up — answers

**1. How does Pipeline + `cross_val_score` prevent leakage?**  
Each CV fold creates a separate training/validation split. Pipeline transformers are fitted on the training fold and then used to transform the held-out fold.

**2. Where else can leakage happen?**  
Target encoding, imputation, feature selection, scaling, normalization, and feature engineering that uses future information can all leak.

**3. If you inherit a notebook, what should you audit first?**  
Look for preprocessing operations performed before train/test splitting or outside cross-validation, especially anything using target values or statistics calculated from all rows.

**Important:** A leaky score can sometimes be very close to the fixed score. That does not make the method correct. The issue is the preprocessing order, not whether the numerical gap happens to be large.

**Submission evidence:** Both versions and their actual CV scores are printed.


In [8]:
# TASK 06 — Full ColumnTransformer + Pipeline

# Custom date transformer function.
# It returns numeric date features.
# The final custom transformer in Task 07 shows the same idea
# with an explicitly learned training reference.

def make_date_features(X):
    X = pd.DataFrame(X).copy()

    dates = pd.to_datetime(
        X.iloc[:, 0],
        errors="coerce"
    )

    # Within a CV fold this reference is computed only from the
    # data passed to this transformer during fit/transform.
    reference = dates.max()

    return pd.DataFrame({
        "signup_year": dates.dt.year,
        "signup_month": dates.dt.month,
        "signup_dayofweek": dates.dt.dayofweek,
        "signup_quarter": dates.dt.quarter,
        "days_since": (reference - dates).dt.days
    }, index=X.index)


# Numeric branch
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical branch
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Date branch
date_pipeline = Pipeline([
    ("date_features", FunctionTransformer(
        make_date_features,
        validate=False
    )),
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

prep = ColumnTransformer([
    ("num", numeric_pipeline, numeric_cols),
    ("cat", categorical_pipeline, categorical_cols),
    ("date", date_pipeline, [date_col])
], remainder="drop")


full_pipeline = Pipeline([
    ("prep", prep),
    ("clf", LogisticRegression(max_iter=3000))
])


full_scores = cross_val_score(
    full_pipeline,
    X,
    y,
    cv=cv,
    scoring="accuracy"
)

print("Full Pipeline CV scores:")
print(full_scores)

print("\nFull Pipeline mean CV accuracy:")
print(full_scores.mean())

print("\nFull Pipeline std CV accuracy:")
print(full_scores.std())

best_single_score = max(
    onehot_score,
    ordinal_score,
    target_score,
    scaler_scores["StandardScaler"],
    scaler_scores["MinMaxScaler"],
    scaler_scores["RobustScaler"],
    base_cv,
    poly_cv,
    interaction_cv,
    date_cv
)

print("\nBest single-task comparison score:")
print(best_single_score)

print("\nFull pipeline - best single-task:")
print(full_scores.mean() - best_single_score)


NameError: name 'FunctionTransformer' is not defined

## Task 06 — Final ColumnTransformer + Pipeline

### Pipeline structure

- Numeric → median imputation → StandardScaler
- Categorical → most-frequent imputation → OneHotEncoder
- Date → date feature extraction → imputation → StandardScaler
- Everything → `ColumnTransformer`
- Model → LogisticRegression
- Evaluation → cross-validation

### Look it up — answers

**1. What happens to a column left out of all transformers?**  
With `remainder="drop"`, it is removed from the transformed data.

**2. Why should `customer_id` be dropped instead of encoded?**  
It is an identifier, not a meaningful behavioral feature. Encoding it can create meaningless categories and encourage memorization.

**3. How does one Pipeline guarantee the test set is not touched before scoring?**  
During cross-validation, sklearn fits the Pipeline on the training fold. Every transformer inside it is therefore fitted only on that training fold before the validation fold is scored.

**Required deliverable:** One `ColumnTransformer + Pipeline` processes the full mixed-type DataFrame and is evaluated with cross-validation.

**Important interpretation:** The task asks the full pipeline to beat the best single-task score. If the actual dataset does not produce that result, report the real result rather than changing the score.


In [ ]:
# TASK 07 — Custom transformer

class DateFeatureExtractor(BaseEstimator, TransformerMixin):

    def __init__(self):
        self.reference_date_ = None

    def fit(self, X, y=None):
        X = pd.DataFrame(X).copy()

        dates = pd.to_datetime(
            X.iloc[:, 0],
            errors="coerce"
        )

        # Learned only from the training fold
        self.reference_date_ = dates.max()

        return self

    def transform(self, X):
        X = pd.DataFrame(X).copy()

        dates = pd.to_datetime(
            X.iloc[:, 0],
            errors="coerce"
        )

        output = pd.DataFrame({
            "signup_year": dates.dt.year,
            "signup_month": dates.dt.month,
            "signup_dayofweek": dates.dt.dayofweek,
            "signup_quarter": dates.dt.quarter,
            "days_since": (
                self.reference_date_ - dates
            ).dt.days
        }, index=X.index)

        return output


custom_prep = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]), numeric_cols),

    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]), categorical_cols),

    ("date", Pipeline([
        ("custom_date", DateFeatureExtractor()),
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]), [date_col])
])


custom_pipeline = Pipeline([
    ("prep", custom_prep),
    ("clf", LogisticRegression(max_iter=3000))
])


custom_scores = cross_val_score(
    custom_pipeline,
    X,
    y,
    cv=cv,
    scoring="accuracy"
)

print("Custom transformer CV scores:")
print(custom_scores)

print("\nCustom transformer mean CV accuracy:")
print(custom_scores.mean())

print("\nCustom transformer std CV accuracy:")
print(custom_scores.std())


# Demonstrate learned state
demo = DateFeatureExtractor()

demo.fit(
    X_train[[date_col]]
)

print("\nReference date learned from training data:")
print(demo.reference_date_)

print("\nTransformed sample:")
print(
    demo.transform(
        X_train[[date_col]]
    ).head()
)


Custom transformer CV scores:
[0.7775 0.7725 0.775  0.77   0.7725]

Custom transformer mean CV accuracy:
0.7735

Custom transformer std CV accuracy:
0.0025495097567963857

Reference date learned from training data:
2023-12-31 00:00:00

Transformed sample:
      signup_year  signup_month  signup_dayofweek  signup_quarter  days_since
1604         2021             9                 6               3         840
1417         2023             1                 0               1         356
184          2021            10                 4               4         793
196          2023             5                 0               2         216
1806         2021             7                 0               3         909


## Task 07 — Custom transformer

The custom `DateFeatureExtractor` inherits from `BaseEstimator` and `TransformerMixin`.

- `fit()` learns the reference date.
- `transform()` creates useful numeric date features.
- The transformer is inside `ColumnTransformer`.
- The `ColumnTransformer` is inside a `Pipeline`.
- Therefore, cross-validation fits the transformer separately on each training fold.

### Look it up — answers

**1. What must `fit` and `transform` return?**  
`fit()` should return `self`, allowing sklearn to chain estimators. `transform()` should return the transformed data.

**2. When would you implement `inverse_transform`?**  
When it is useful to convert transformed data back into the original representation, for example reversing a scaling or encoding transformation.

**3. How does `cross_val_score` know to call only `fit` on each training fold?**  
It clones the estimator for each fold, fits it on the training portion, and evaluates it on the held-out portion. Since the custom transformer is inside the Pipeline, its `fit()` follows the same fold-specific process.

**Leakage check:** The reference date is learned inside `fit()` and reused during `transform()`. It is not learned from the validation/test fold.

**Submission evidence:** The code prints CV scores and demonstrates the learned training reference date.
